In [ ]:
import os,sys
import re
import numpy as np
import pandas as pd
import scanpy as sc
from scanpy import AnnData
import mudata
from scipy.sparse import csr_matrix
import warnings
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import harmonypy as hm

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
sc.settings.verbosity = 3
warnings.filterwarnings("ignore")
plt.rcParams['pdf.fonttype']=42

In [ ]:
#second trimester, better age match

snm3_2t = sc.read_h5ad("/u/project/cluo/heffel/BICAN3/DATA/All_2T_61842x49367.h5ad")
snm3_2t

In [ ]:
#resegmented, with min 100 umi count filter

bgs4 = sc.read("/u/project/cluo/mbaig/cosmx/analyses/20250527_MB_basal_2/processed_data/20260106/MBbrain6Kbasalslide4_reseg_normalized1.h5ad")

In [ ]:
# processing bgs4
from matplotlib.path import Path

spatial_coords = bgs4.obsm['spatial']
polygon_coords = np.array([
    [5400, 14000],
    [5100, 12000],
    [5000, 11000],
    [5000, 10000],
    [5300, 8000],
    [5500, 7000],
    [5000, 6000],
    [5000, 5000],
    [7200, 1800],
    [8000, 6000],
    [8000, 6500],
    [8300, 8000],
    [9000, 9000],
    [10100, 11000],
    [9800, 13500]
])

# Create mask
polygon_path = Path(polygon_coords)
inside_polygon = polygon_path.contains_points(spatial_coords)
bgs4_lge_mge = bgs4[inside_polygon, :].copy()

In [ ]:
bgs4_lge_mge

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Get the cluster annotation
clusters = bgs4_lge_mge.obs['leiden']

# Convert to string for hue (safe handling)
cluster_str = clusters.astype(str)

# Identify numeric clusters for sorting
numeric = pd.to_numeric(clusters, errors='coerce')
is_numeric = numeric.notna()
numeric_unique = np.sort(numeric[is_numeric].unique().astype(int))

# Non-numeric clusters
non_numeric_unique = np.sort(clusters[~is_numeric].unique())

# Full sorted list: numerics first (as str), then non-numerics
all_unique = [str(c) for c in numeric_unique] + [str(c) for c in non_numeric_unique]

# Color palette (tab20 has 20 colors; increase n if you have more clusters)
palette = sns.color_palette('tab20', len(all_unique))
cluster_colors = {clust: palette[i] for i, clust in enumerate(all_unique)}

# Spatial coordinates - prefer obsm['spatial'] if present
if 'spatial' in bgs4_lge_mge.obsm:
    spatial_coords = bgs4_lge_mge.obsm['spatial']
    x_spatial, y_spatial = spatial_coords[:, 0], spatial_coords[:, 1]
else:
    x_spatial = bgs4_lge_mge.obs['CenterX_global_px']
    y_spatial = bgs4_lge_mge.obs['CenterY_global_px']

# UMAP coordinates
umap_coords = bgs4_lge_mge.obsm['X_umap']

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Spatial plot
sns.scatterplot(x=x_spatial, y=y_spatial,
                hue=cluster_str, palette=cluster_colors,
                s=2, edgecolor=None, ax=axes[0], legend=False)
axes[0].set_title('Leiden - Spatial')
axes[0].set_xlabel('X (px)')
axes[0].set_ylabel('Y (px)')
axes[0].invert_yaxis()  # Important for aligning with original images
axes[0].set_aspect('equal')

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=cluster_str, palette=cluster_colors,
                s=5, edgecolor=None, ax=axes[1], legend=False)
axes[1].set_title('Leiden - UMAP')
axes[1].set_xlabel('UMAP 1')
axes[1].set_ylabel('UMAP 2')

# Shared legend outside
handles = [plt.Line2D([0], [0], marker='o', color='w',
                      markerfacecolor=cluster_colors[clust],
                      markersize=10) for clust in all_unique]
fig.legend(handles=handles, labels=all_unique,
           title='Leiden',
           bbox_to_anchor=(1.02, 0.5), loc='center left')

plt.tight_layout(rect=[0, 0, 0.90, 1])
plt.show()

#true cosmx leidens

In [ ]:
kept_L2 = ['Glial', 'MSN', 'NN']

mask = snm3_2t.obs['newL2'].isin(kept_L2)

snm3_2t_filt = snm3_2t[mask, :]

In [ ]:
snm3_2t_filt.X = snm3_2t_filt.X * -1

In [ ]:
sc.pp.scale(snm3_2t_filt)

In [ ]:
sc.pp.scale(bgs4_lge_mge)

In [ ]:
harmonized_2t = bgs4_lge_mge.concatenate(snm3_2t_filt)

In [ ]:
sc.tl.pca(harmonized_2t)

In [ ]:
sc.external.pp.harmony_integrate(harmonized_2t, 'batch', max_iter_harmony=20)

In [ ]:
sc.pp.neighbors(harmonized_2t, n_pcs=20, n_neighbors=20, use_rep='X_pca_harmony')

In [ ]:
sc.tl.umap(harmonized_2t, min_dist=0.3, spread=1.2)

In [ ]:
sc.tl.leiden(harmonized_2t, key_added = 'h_leiden')

In [ ]:
sc.pl.umap(harmonized_2t, color = ['h_leiden'], legend_loc = 'on data')

#bgs4_reseg

In [ ]:
# Rename batch categories for plotting
harmonized_2t.obs['batch_named'] = harmonized_2t.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

sc.pl.umap(
    harmonized,
    color='batch_named',
    size=1.5,
    palette={
        'CosMx': '#4E79A7',   # muted steel blue
        'snm3C': '#F28E2B'    # soft orange
    }
)

In [ ]:
#shuffled order --> dont want

import numpy as np
import scanpy as sc

# Rename batch categories for plotting
harmonized_2t.obs['batch_named'] = harmonized_2t.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

# Shuffle plotting order (reproducible with a seed)
rng = np.random.default_rng(0)
perm = rng.permutation(harmonized_2t.n_obs)

# Plot using the shuffled view
sc.pl.umap(
    harmonized_2t[perm],
    color='batch_named',
    size=2,
    palette={
        'CosMx': 'tab:blue',
        'snm3C': 'tab:orange'
    }
)

In [ ]:
import scanpy as sc
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
import pandas as pd

def knn_label_transfer_from_harmony(
    integrated_adata,
    target_adata,
    label_key='cell_type',
    batch_key='batch',
    target_batch=None,
    harmony_key='X_pca_harmony',
    n_neighbors=15,
    use_rep='X_pca',
    confidence_threshold=0.5,
    return_probabilities=True
):
    """
    Transfer labels from harmony-integrated object to original object using KNN.

    Parameters:
    -----------
    integrated_adata : AnnData
        Concatenated and harmony-integrated object with labels
    target_adata : AnnData
        Original object to receive transferred labels
    label_key : str
        Key in integrated_adata.obs containing labels to transfer
    batch_key : str
        Key identifying different batches/samples
    target_batch : str
        Batch identifier for the target dataset in integrated object
    harmony_key : str
        Key for harmony-corrected embedding in integrated_adata.obsm
    n_neighbors : int
        Number of neighbors for KNN classifier
    use_rep : str
        Representation to use in target_adata (will be computed if needed)
    confidence_threshold : float
        Minimum probability to assign label (otherwise 'uncertain')
    return_probabilities : bool
        Whether to return prediction probabilities

    Returns:
    --------
    target_adata with transferred labels in .obs
    """

    # Get harmony-corrected embedding from integrated object
    if harmony_key not in integrated_adata.obsm:
        raise ValueError(f"{harmony_key} not found in integrated_adata.obsm")

    X_harmony = integrated_adata.obsm[harmony_key]

    # Identify reference cells (all cells except target batch)
    if target_batch is None:
        # If not specified, assume target_adata represents a specific batch
        # Use all cells in integrated object as reference
        ref_mask = np.ones(integrated_adata.n_obs, dtype=bool)
        target_mask = np.zeros(integrated_adata.n_obs, dtype=bool)
        print("Warning: target_batch not specified. Using all integrated cells as reference.")
        print("For proper transfer, specify which batch in integrated object corresponds to target_adata")
    else:
        ref_mask = integrated_adata.obs[batch_key] != target_batch
        target_mask = integrated_adata.obs[batch_key] == target_batch

    # Get reference data and labels
    X_ref = X_harmony[ref_mask]
    y_ref = integrated_adata.obs[label_key].values[ref_mask]

    # Get target cells from integrated object (for matching)
    X_target_integrated = X_harmony[target_mask]

    # Train KNN classifier on reference cells
    print(f"Training KNN classifier with {n_neighbors} neighbors...")
    knn = KNeighborsClassifier(n_neighbors=n_neighbors, weights='distance')
    knn.fit(X_ref, y_ref)

    # Predict labels for target cells in integrated space
    y_pred = knn.predict(X_target_integrated)
    y_proba = knn.predict_proba(X_target_integrated)

    # Get maximum probability for each prediction
    max_proba = y_proba.max(axis=1)

    # Apply confidence threshold
    y_pred_confident = y_pred.copy()
    uncertain_mask = max_proba < confidence_threshold
    if uncertain_mask.sum() > 0:
        y_pred_confident[uncertain_mask] = 'uncertain'
        print(f"Marked {uncertain_mask.sum()} cells as uncertain (confidence < {confidence_threshold})")

    # Match cells between integrated object and original target object
    # This assumes same order or we can match by cell barcodes
    if target_adata.n_obs == target_mask.sum():
        # Same number of cells - assume same order
        target_adata.obs[f'{label_key}_transferred'] = y_pred_confident
        target_adata.obs[f'{label_key}_confidence'] = max_proba

        if return_probabilities:
            # Store full probability matrix
            proba_df = pd.DataFrame(
                y_proba,
                index=target_adata.obs_names,
                columns=[f'{label_key}_prob_{cls}' for cls in knn.classes_]
            )
            for col in proba_df.columns:
                target_adata.obs[col] = proba_df[col].values
    else:
        print(f"Warning: target_adata has {target_adata.n_obs} cells but "
              f"found {target_mask.sum()} cells with batch={target_batch} in integrated object")
        print("Cell matching may be incorrect. Consider matching by cell barcodes.")

    print(f"\nLabel transfer complete!")
    print(f"Transferred labels stored in: {label_key}_transferred")
    print(f"Confidence scores stored in: {label_key}_confidence")

    # Print summary statistics
    print(f"\nTransferred label distribution:")
    print(target_adata.obs[f'{label_key}_transferred'].value_counts())
    print(f"\nMean confidence: {max_proba.mean():.3f}")
    print(f"Median confidence: {np.median(max_proba):.3f}")

    return target_adata

In [ ]:
# If you have integrated object with harmony correction
knn_label_transfer_from_harmony(
    integrated_adata=harmonized_2t,  # your harmony-corrected concatenated object
    target_adata=bgs4_lge_mge,  # your original CosMx spatial data
    label_key='adjusted_L3',
    batch_key='batch',
    target_batch='0',  # whatever identifier you used
    harmony_key='X_pca_harmony',
    n_neighbors=20,  # adjust based on your data
    confidence_threshold=0.6
)

# Now your spatial object has transferred labels you can visualize
sc.pl.umap(bgs4_lge_mge, color='adjusted_L3_transferred')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

adata = bgs4_lge_mge
categ = 'adjusted_L3_transferred'

# Get unique clusters
clusters = sorted(adata.obs[categ].unique())
n_clusters = len(clusters)

# Grid setup
ncols = 4
nrows = int(np.ceil(n_clusters / ncols))
figsize = (6 * ncols, 6 * nrows)

# Create figure
fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
axes = axes.flatten() if n_clusters > 1 else [axes]

# Get spatial coordinates
spatial_coords = adata.obsm['spatial']

# Get colors for clusters
if f'{categ}_colors' in adata.uns:
    cluster_colors = dict(zip(adata.obs[categ].cat.categories, adata.uns[f'{categ}_colors']))
else:
    from matplotlib import cm
    cluster_colors = {c: cm.tab20(i % 20) for i, c in enumerate(clusters)}

grey_color = '#D3D3D3'
spot_size = 0.5

# Plot each cluster
for idx, cluster in enumerate(clusters):
    ax = axes[idx]

    # Create mask for this cluster
    mask = adata.obs[categ] == cluster

    # Plot grey background (all other cells)
    ax.scatter(
        spatial_coords[~mask, 0],
        spatial_coords[~mask, 1],
        c=grey_color,
        s=spot_size,
        alpha=0.5
    )

    # Plot this cluster in color
    ax.scatter(
        spatial_coords[mask, 0],
        spatial_coords[mask, 1],
        c=cluster_colors[cluster],
        s=spot_size,
        alpha=1.0
    )

    ax.set_title(f'Leiden {cluster}')
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.axis('off')

# Remove empty subplots
for idx in range(n_clusters, len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
#plt.savefig('individual_clusters_spatial.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import os

outdir = "/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20260106/"
os.makedirs(outdir, exist_ok=True)

# Save AnnData objects
bgs4_lge_mge.write(os.path.join(outdir, "20260107_bgs4_reseg_snm3_2T_L3.h5ad"))

In [ ]:
del bgs4

In [ ]:
bgs4 = bgs4_lge_mge.copy()
del bgs4_lge_mge
bgs4

In [ ]:
bgs5 = sc.read_h5ad("/u/project/cluo/rayirfan/cosmx/analyses/basalganglia/D_snm3/bgs5_snm3_3T_L2_L3.h5ad")

#3T --> bgs5

bgs5

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.sparse as sp

# Two separate heatmaps: marker gene expression across cell types
# - Titles: "2T  GW24 (bgs4)" and "3T  GW35 (bgs5)"
# - Z-score clipped to [-2, 2]
# - Cell counts shown on the right
# - Fully customizable: genes + separate cell type lists for each dataset

# 1. Explicitly define genes to show (in desired order)
custom_gene_order = [
    'DRD1',      # Direct MSN marker
    'DRD2',      # Indirect MSN marker

    'BACH2',     # Transcription factor (DRD1 subtype)
    'EPHA4',     # Eph receptor (DRD1/DRD2 subtype)
    'PENK',      # Enkephalin - classic D2-MSN marker (indirect pathway)
    'GAD1',      # GABAergic interneuron marker

    'PAX6',
    #'PDGFRA',
    'OLIG1',
    #'SOX2',
    #'NES',

    'OLIG2',     # Oligodendrocyte lineage
    'SOX10',     # Oligodendrocyte lineage
    'PDGFRA',     # Strong OPC and pericyte marker (careful overlap with PC)
    'MBP',        # Myelin basic protein - mature oligodendrocyte marker

    'ALDH1L1',   # Astrocyte marker
    'GFAP',      # Astrocyte marker
    'AQP4',      # Astrocyte marker

    'ESAM',      # Endothelial marker
    'FN1',       # Endothelial / pericyte
    'LUM',        # Lumican - top VLMC/fibroblast-like marker
    'DCN',        # Decorin - VLMC marker

    'P2RY12',    # Microglia marker
    'CX3CR1',     # Fractalkine receptor - microglia marker (complements P2RY12)
    #'SLC17A7',    # VGLUT1 - excitatory neuron marker (useful for any glutamatergic contamination or comparison)

]

# 2. Explicitly define cell types to KEEP for EACH dataset
# Edit these lists independently - only types in the list will appear in that heatmap
cell_types_bgs4 = [          # 2T  GW24
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',

    'GPC',
    'OPC',
    #'PAX6-TCx',
    'Astro',

    'EC',
    'PC',
    'MGC-1',
    # 'uncertain',  # excluded
]

cell_types_bgs5 = [          # 3T  GW35
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1',
    'CABLES1',

    'GPC',
    'ODC',
    'OPC',
    'Astro',

    #'PVALB',
    #'SST',
    'EC',
    'PC',
    'VLMC',
    'MGC-1',
    # 'uncertain',  # excluded
]

# 3. Filtering and area-based QC
label_col = 'adjusted_L3_transferred'
min_area_um2 = 10
um_per_pixel = 0.12028

bgs4_f = bgs4.copy()
bgs5_f = bgs5.copy()

# for adata in [bgs4_f, bgs5_f]:
#     nuc_area = adata.obs['NucArea'] * (um_per_pixel ** 2)
#     cell_area = adata.obs['Area'] * (um_per_pixel ** 2)
#     mask = (nuc_area >= min_area_um2) & (cell_area >= min_area_um2)
#     adata = adata[mask].copy()

# 4. Restrict to desired cell types for each dataset separately
bgs4_f = bgs4_f[bgs4_f.obs[label_col].isin(cell_types_bgs4)].copy()
bgs5_f = bgs5_f[bgs5_f.obs[label_col].isin(cell_types_bgs5)].copy()

# 5. Pseudobulk z-score function
def pseudobulk_zscore_fine(adata, genes, groupby=label_col):
    genes_use = [g for g in genes if g in adata.var_names]
    print(f"{adata.uns.get('name','adata')} - using {len(genes_use)}/{len(genes)} genes")

    if 'counts_downsampled' in adata.layers:
        X = adata[:, genes_use].layers['counts_downsampled']
    else:
        X = adata[:, genes_use].X

    if sp.issparse(X):
        X = X.toarray()

    df = pd.DataFrame(X, columns=genes_use)
    df[groupby] = adata.obs[groupby].values
    pb = df.groupby(groupby).mean()

    pb_z = (pb - pb.mean(axis=0)) / (pb.std(axis=0) + 1e-8)
    pb_z = pb_z.fillna(0)
    pb_z = pb_z.clip(lower=-2, upper=2)

    return pb_z, genes_use

# 6. Compute pseudobulk for each dataset
pb4, genes4 = pseudobulk_zscore_fine(bgs4_f, custom_gene_order)
pb5, genes5 = pseudobulk_zscore_fine(bgs5_f, custom_gene_order)

# Use only genes present in BOTH datasets (for visual alignment)
shared_genes = [g for g in custom_gene_order if g in genes4 and g in genes5]
pb4 = pb4.loc[:, shared_genes]
pb5 = pb5.loc[:, shared_genes]

# 6b. Reorder cell types in pseudobulk according to custom order
pb4 = pb4.reindex(cell_types_bgs4)
pb5 = pb5.reindex(cell_types_bgs5)

# 7. Cell counts
counts4 = bgs4_f.obs[label_col].value_counts().reindex(pb4.index).fillna(0).astype(int)
counts5 = bgs5_f.obs[label_col].value_counts().reindex(pb5.index).fillna(0).astype(int)

# 8. Plotting
fig = plt.figure(figsize=(14, (len(pb4) + len(pb5)) * 0.32 + 6))
gs = fig.add_gridspec(2, 1, hspace=0.2, height_ratios=[len(pb4), len(pb5)])

# Top: 2T  GW24
ax1 = fig.add_subplot(gs[0])
g1 = sns.heatmap(pb4, cmap='vlag', vmin=-2, vmax=2, center=0,
                 linewidths=0.5, linecolor='lightgray',
                 cbar_kws={"orientation": "horizontal", "pad": 0.2, "shrink": 0.6},  # <- increase pad
                 ax=ax1)
ax1.set_title("2T  GW24 (bgs4) - Selected Cell Types (pseudobulk z-scored, clipped [-2, 2])",
              fontsize=15, pad=12)
ax1.set_ylabel("Cell Type", fontsize=13)
ax1.set_xlabel("")
ax1.tick_params(axis='x', rotation=90, labelsize=11)
cbar1 = g1.collections[0].colorbar
cbar1.set_label('z-score', fontsize=11)

# Cell counts (right side)
for i, n in enumerate(counts4):
    ax1.text(len(shared_genes) + 0.25, i + 0.5, f"n={n:,}", va='center', ha='left', fontsize=9)
ax1.axvline(len(shared_genes), color='black', linewidth=1.2, linestyle='--')

# Bottom: 3T  GW35
ax2 = fig.add_subplot(gs[1])
g2 = sns.heatmap(pb5, cmap='vlag', vmin=-2, vmax=2, center=0,
                 linewidths=0.5, linecolor='lightgray',
                 cbar_kws={"orientation": "horizontal", "pad": 0.2, "shrink": 0.6},  # <- increase pad
                 ax=ax2)
ax2.set_title("3T  GW35 (bgs5) - Selected Cell Types (pseudobulk z-scored, clipped [-2, 2])",
              fontsize=15, pad=12)
ax2.set_ylabel("Cell Type", fontsize=13)
ax2.set_xlabel("Genes", fontsize=13)
ax2.tick_params(axis='x', rotation=90, labelsize=11)
cbar2 = g2.collections[0].colorbar
cbar2.set_label('z-score', fontsize=11)

# Cell counts (right side)
for i, n in enumerate(counts5):
    ax2.text(len(shared_genes) + 0.25, i + 0.5, f"n={n:,}", va='center', ha='left', fontsize=9)
ax2.axvline(len(shared_genes), color='black', linewidth=1.2, linestyle='--')

plt.suptitle("\nMarker Gene Expression Across Selected Cell Types\n2T  GW24 (top) vs 3T  GW35 (bottom)",
             fontsize=17, y=0.985)

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl

# STYLE
mpl.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 16,
    'legend.fontsize': 14,
})

# PARAMETERS
min_area_um2 = 10
um_per_pixel = 0.12028
nuc_area_col = 'NucArea'
cell_area_col = 'Area'
label_col = 'adjusted_L3_transferred'

# PREPARE DATA
df4 = bgs4.obs[[label_col, nuc_area_col, cell_area_col]].copy()
df4['Dataset'] = 'GW24'  # Corrected label
df5 = bgs5.obs[[label_col, nuc_area_col, cell_area_col]].copy()
df5['Dataset'] = 'GW35'

combined = pd.concat([df4, df5], ignore_index=True)

# Clean labels: drop NaN, '?' and 'uncertain'
combined = combined.dropna(subset=[label_col])
combined = combined[combined[label_col] != '?']
combined = combined[combined[label_col] != 'uncertain']

# Convert to µm²
combined['NucArea_um2'] = combined[nuc_area_col] * (um_per_pixel ** 2)
combined['CellArea_um2'] = combined[cell_area_col] * (um_per_pixel ** 2)

# Filter small objects
combined = combined[(combined['NucArea_um2'] >= min_area_um2) & (combined['CellArea_um2'] >= min_area_um2)]

# FIND COMMON CELL TYPES
types_gw24 = set(combined[combined['Dataset'] == 'GW24'][label_col].unique())
types_gw35 = set(combined[combined['Dataset'] == 'GW35'][label_col].unique())
common_types = sorted(types_gw24.intersection(types_gw35))  # alphabetical

# Restrict to common types only
combined = combined[combined[label_col].isin(common_types)]

# MEDIAN AREAS
medians = combined.groupby(['Dataset', label_col])[ ['NucArea_um2', 'CellArea_um2'] ].median().reset_index()

# Ensure alphabetical order
medians[label_col] = pd.Categorical(medians[label_col], categories=common_types, ordered=True)
medians = medians.sort_values([label_col, 'Dataset'])

# PLOTTING: Two side-by-side plots
fig, axes = plt.subplots(1, 2, figsize=(24, len(common_types) * 0.7 + 4), sharey=True)

# Nuclear Area (Median)
sns.barplot(
    data=medians,
    y=label_col,
    x='NucArea_um2',
    hue='Dataset',
    order=common_types,
    palette={'GW24': '#1f77b4', 'GW35': '#ff7f0e'},
    ax=axes[0]
)
axes[0].set_title('Median Nuclear Area', fontsize=22)
axes[0].set_xlabel('Median Nuclear Area (µm²)')
axes[0].set_ylabel('Cell Type')
axes[0].set_xscale('log')
axes[0].legend(title='Stage', loc='lower right')

# Cell Area (Median)
sns.barplot(
    data=medians,
    y=label_col,
    x='CellArea_um2',
    hue='Dataset',
    order=common_types,
    palette={'GW24': '#1f77b4', 'GW35': '#ff7f0e'},
    ax=axes[1]
)
axes[1].set_title('Median Cell Area', fontsize=22)
axes[1].set_xlabel('Median Cell Area (µm²)')
axes[1].set_ylabel('')
axes[1].set_xscale('log')
axes[1].legend_.remove()

plt.suptitle('Basal Ganglia L3 Cell Types Present in Both Stages\n'
             'Median Nuclear and Cell Areas (GW24 vs GW35)',
             fontsize=24, y=0.98)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

# SUMMARY TABLE (medians, common types only)
summary_medians = medians.pivot(index=label_col, columns='Dataset',
                                values=['NucArea_um2', 'CellArea_um2'])
summary_medians.columns = ['_'.join(col).strip() for col in summary_medians.columns]
summary_medians = summary_medians.round(1)
summary_medians = summary_medians.sort_index()

print("\nSummary Table: Median Areas (µm²) - Common Cell Types Only (excluding 'uncertain')")
print(summary_medians)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl

# STYLE
mpl.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 16,
    'legend.fontsize': 14,
})

# PARAMETERS
min_area_um2 = 10
um_per_pixel = 0.12028
nuc_area_col = 'NucArea'
cell_area_col = 'Area'
label_col = 'adjusted_L3_transferred'

# PREPARE DATA
df4 = bgs4.obs[[label_col, nuc_area_col, cell_area_col]].copy()
df4['Dataset'] = 'GW24 (2T)'  # Corrected label
df5 = bgs5.obs[[label_col, nuc_area_col, cell_area_col]].copy()
df5['Dataset'] = 'GW35 (3T)'

combined = pd.concat([df4, df5], ignore_index=True)

# Remove uncertain / unknown labels
combined = combined.dropna(subset=[label_col])
combined = combined[combined[label_col] != '?']
combined = combined[combined[label_col] != 'uncertain']

# Convert to µm²
combined['NucArea_um2'] = combined[nuc_area_col] * (um_per_pixel ** 2)
combined['CellArea_um2'] = combined[cell_area_col] * (um_per_pixel ** 2)

# Filter small objects
combined = combined[(combined['NucArea_um2'] >= min_area_um2) & (combined['CellArea_um2'] >= min_area_um2)]

# MEDIAN VALUES & LOG2 FOLD CHANGE
medians = combined.groupby(['Dataset', label_col])[ ['NucArea_um2', 'CellArea_um2'] ].median()

gw24_medians = medians.loc['GW24 (2T)']
gw35_medians = medians.loc['GW35 (3T)']

# Only keep cell types present in both datasets
common_types = gw24_medians.index.intersection(gw35_medians.index)
gw24_medians = gw24_medians.loc[common_types]
gw35_medians = gw35_medians.loc[common_types]

# Compute log2 fold change (GW35 / GW24)
fc_nuc = np.log2(gw35_medians['NucArea_um2'] / gw24_medians['NucArea_um2'])
fc_cell = np.log2(gw35_medians['CellArea_um2'] / gw24_medians['CellArea_um2'])

# Alphabetical order
order = sorted(common_types)

# PLOTTING
fig, axes = plt.subplots(1, 2, figsize=(20, len(order) * 0.65 + 5), sharey=True)

# Nuclear Area Log2 Fold Change (median-based)
sns.barplot(x=fc_nuc[order], y=order, color='#2ca02c', ax=axes[0])
axes[0].set_title('Nuclear Area', fontsize=20)
axes[0].set_xlabel('Log₂ Fold Change (median)')
axes[0].set_ylabel('Cell Type')
axes[0].axvline(0, color='gray', linewidth=1, linestyle='--')

# Cell Area Log2 Fold Change (median-based)
sns.barplot(x=fc_cell[order], y=order, color='#d62728', ax=axes[1])
axes[1].set_title('Cell Area', fontsize=20)
axes[1].set_xlabel('Log₂ Fold Change (median)')
axes[1].set_ylabel('')
axes[1].axvline(0, color='gray', linewidth=1, linestyle='--')

# Main title
plt.suptitle('Median Size Changes\n'
             'Log₂ Fold Change of Median Areas (GW35 ÷ GW24)\n'
             '<---GW24 larger          GW35 larger--->',
             fontsize=24, y=1.00)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# SUMMARY TABLE
summary = pd.DataFrame({
    'Median_Nuc_GW24': gw24_medians['NucArea_um2'].round(1),
    'Median_Nuc_GW35': gw35_medians['NucArea_um2'].round(1),
    'Log2FC_Nuclear': fc_nuc.round(3),
    'Median_Cell_GW24': gw24_medians['CellArea_um2'].round(1),
    'Median_Cell_GW35': gw35_medians['CellArea_um2'].round(1),
    'Log2FC_Cell': fc_cell.round(3),
}).loc[order]

print("\nSummary Table (median-based, alphabetical order, common cell types only):")
print(summary)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

# SINGLE FIGURE (UPDATED for bgs4_lge_mge + harmonized_2t):
# - Left: CosMx spatial (bgs4_lge_mge) colored by transferred cell types
# - Right: Harmonized JOINT embedding UMAP (CosMx cells only), same colors
# - Excludes these cell types entirely:
#     SST, VLMC, PVALB, ChC, uncertain
# Legend bug fixed (labels are plain python list).
# REQUIREMENTS / ASSUMPTIONS:
# 1) harmonized_2t.obsm['X_umap'] exists (sc.tl.umap(harmonized_2t))
# 2) harmonized_2t.obs[batch_key] exists and CosMx batch == target_batch
# 3) CosMx cells in harmonized_2t align to bgs4_lge_mge either by obs_names
#    or by id_col (fallback).

# Objects
adata = bgs4_lge_mge
harm = harmonized_2t

# Settings (edit if needed)
color_key = 'adjusted_L3_transferred'
batch_key = 'batch'
target_batch = '0'          # CosMx batch label inside harm.obs[batch_key]
id_col = 'cell_id'          # fallback matching key if obs_names don't match
exclude_types = {'SST', 'VLMC', 'PVALB', 'ChC', 'uncertain'}

# Plot styling
spatial_s = 1.0
umap_s = 1.5
alpha = 0.85

# Sanity checks
if 'X_umap' not in harm.obsm:
    raise ValueError("harm.obsm['X_umap'] not found. Run sc.tl.umap(harm) first (i.e., sc.tl.umap(harmonized_2t)).")
if color_key not in adata.obs.columns:
    raise KeyError(f"{color_key} not found in adata.obs (bgs4_lge_mge.obs).")
if batch_key not in harm.obs.columns:
    raise KeyError(f"{batch_key} not found in harm.obs (harmonized_2t.obs).")

# Restrict harmonized to CosMx batch (UMAP coords for CosMx only)
harm_cosmx_mask = harm.obs[batch_key].astype(str).values == str(target_batch)
harm_umap_cosmx = harm.obsm['X_umap'][harm_cosmx_mask, :]
harm_cosmx_names = harm.obs_names[harm_cosmx_mask]

# Align harmonized CosMx UMAP rows to adata (bgs4_lge_mge)
if np.array_equal(harm_cosmx_names.values, adata.obs_names.values):
    umap_coords = harm_umap_cosmx
else:
    # Fallback: match by id_col
    if id_col not in adata.obs.columns or id_col not in harm.obs.columns:
        raise ValueError(
            "CosMx cells in harmonized_2t do not match bgs4_lge_mge.obs_names, and "
            f"'{id_col}' is missing from obs in one of the objects. "
            "Either ensure obs_names match OR set id_col to a shared unique ID (e.g. 'cell_ID')."
        )

    harm_cosmx_obs = harm.obs.loc[harm_cosmx_mask, [id_col]].copy()
    harm_cosmx_obs['__umap_i'] = np.arange(harm_cosmx_obs.shape[0])

    adata_obs = adata.obs[[id_col]].copy()
    adata_obs['__adata_i'] = np.arange(adata_obs.shape[0])

    merged = adata_obs.merge(harm_cosmx_obs, on=id_col, how='inner')

    if merged.shape[0] == 0:
        raise ValueError(
            f"No cells matched between bgs4_lge_mge and harmonized_2t CosMx cells using id_col='{id_col}'. "
            "Try switching id_col to 'cell_ID' (or whichever is truly unique and shared)."
        )

    if merged.shape[0] != adata.n_obs:
        print(
            f"Warning: matched {merged.shape[0]:,} / {adata.n_obs:,} bgs4_lge_mge cells to harmonized_2t CosMx cells via '{id_col}'. "
            "Proceeding with matched subset only."
        )

    umap_coords = harm_umap_cosmx[merged['__umap_i'].values, :]
    adata = adata[merged['__adata_i'].values].copy()

# Filter out excluded cell types (apply to BOTH plots)
labels_all = adata.obs[color_key].astype(str)
mask_keep = ~labels_all.isin(exclude_types)

adata = adata[mask_keep].copy()
umap_coords = umap_coords[mask_keep.values, :]

labels = adata.obs[color_key].astype(str)

# Colors (prefer pre-defined)
unique_labels = np.sort(labels.unique())
unique_labels_list = list(map(str, unique_labels))  # ensure plain python list

if (
    color_key in adata.obs and
    adata.obs[color_key].dtype.name == 'category' and
    f'{color_key}_colors' in adata.uns
):
    categories = adata.obs[color_key].cat.categories.astype(str)
    color_list = adata.uns[f'{color_key}_colors']
    label_colors_all = dict(zip(categories, color_list))

    # Restrict mapping to labels actually present
    label_colors = {k: label_colors_all.get(k, 'lightgrey') for k in unique_labels_list}
    print(f"Using pre-defined colors from adata.uns['{color_key}_colors']")
else:
    palette = sns.color_palette('tab20', len(unique_labels_list))
    label_colors = dict(zip(unique_labels_list, palette))

# Spatial coords
if 'spatial' in adata.obsm:
    xy = adata.obsm['spatial']
    x_sp, y_sp = xy[:, 0], xy[:, 1]
else:
    x_sp = adata.obs['CenterX_global_px'].to_numpy()
    y_sp = adata.obs['CenterY_global_px'].to_numpy()

# Plot: single figure, 2 panels
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Spatial (left)
sns.scatterplot(
    x=x_sp, y=y_sp,
    hue=labels.astype(str), palette=label_colors,
    s=spatial_s, edgecolor=None, ax=axes[0], legend=False, alpha=alpha
)
axes[0].set_title("CosMx Spatial (bgs4_lge_mge: transferred cell types)")
axes[0].set_xlabel("X (px)")
axes[0].set_ylabel("Y (px)")
axes[0].invert_yaxis()
axes[0].set_aspect("equal")

# Harmonized UMAP (right) (CosMx only)
sns.scatterplot(
    x=umap_coords[:, 0], y=umap_coords[:, 1],
    hue=labels.astype(str), palette=label_colors,
    s=umap_s, edgecolor=None, ax=axes[1], legend=False, alpha=alpha
)
axes[1].set_title("Harmonized Joint UMAP (CosMx cells only; harmonized_2t)")
axes[1].set_xlabel("UMAP 1")
axes[1].set_ylabel("UMAP 2")

# Shared legend outside (FIXED)
handles = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=label_colors[l], markersize=7, linestyle='None')
    for l in unique_labels_list
]

fig.legend(
    handles=handles,
    labels=unique_labels_list,
    title=color_key,
    bbox_to_anchor=(1.02, 0.5),
    loc='center left'
)

plt.tight_layout(rect=[0, 0, 0.86, 1])
plt.show()

In [ ]:
#saving harmonized_2t

import anndata as ad

# 1) Subset to 0 vars (this changes shape to (n_obs, 0))
harm_slim = harmonized_2t[:, []].copy()

# 2) Drop big matrices / layers / raw
harm_slim.X = None
harm_slim.layers.clear()
harm_slim.raw = None
harm_slim.obsp.clear()
harm_slim.varm.clear()
harm_slim.uns = {}

# 3) Keep only UMAP embedding (and optionally other embeddings you want)
harm_slim.obsm = {'X_umap': harmonized_2t.obsm['X_umap'].copy()}

# 4) Write
harm_slim.write("20260114_bgs4_2T_harmonized_joint_umap_slim.h5ad", compression="gzip")

In [ ]:
kept_L2 = ['Glial', 'MSN', 'NN']

#kept_L2 = ['Glial', 'MSN', 'NN', 'MGE'] <--important change, did not keep 'MGE' L2 label, better to rename '?' in L3 labels to eMGE instead
#kept_case = ['UCSF2301', 'UCSF2302', 'UCSF2303'] <-- also dropped case 2302

kept_case = ['UCSF2301', 'UCSF2303']

#mask = snm3_2t.obs['newL2'].isin(kept_L2)

mask = snm3_2t.obs['newL2'].isin(kept_L2) & snm3_2t.obs['Case'].isin(kept_case)

snm3_2t_filt = snm3_2t[mask, :]

In [ ]:
sc.pl.umap(harmonized_2t, color = ['h_leiden'], legend_loc = 'on data')

#bgs4 with only cases 'UCSF2301', 'UCSF2302', and 'UCSF2303', plus '?' not MGE directly

In [ ]:
harmonized = harmonized_2t

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc
import math

# GLOBAL FONT SETTINGS
mpl.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 12,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

# Ensure batch names exist
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

# Highlight CosMx (bgs4-derived) cells by LEIDEN
leiden_col = 'leiden'
cosmx_mask = harmonized.obs['batch_named'] == 'CosMx'
m3c_mask   = harmonized.obs['batch_named'] == 'snm3C'

# Keep only CosMx cells that actually have a leiden label
cosmx = harmonized[cosmx_mask].copy()
cosmx = cosmx[~cosmx.obs[leiden_col].isna()].copy()
cosmx.obs[leiden_col] = cosmx.obs[leiden_col].astype('category')
cosmx.obs[leiden_col] = cosmx.obs[leiden_col].cat.remove_unused_categories()
leiden_clusters = list(cosmx.obs[leiden_col].cat.categories)

# Build leiden palette from bgs4 if available, and only keep matching clusters
leiden_palette = {}
if 'leiden_colors' in bgs4.uns:
    bgs4_leiden_cats = (list(bgs4.obs[leiden_col].cat.categories)
                        if hasattr(bgs4.obs[leiden_col], "cat")
                        else sorted(bgs4.obs[leiden_col].dropna().unique()))
    bgs4_leiden_cols = list(bgs4.uns['leiden_colors'])
    bgs4_leiden_palette = dict(zip(bgs4_leiden_cats, bgs4_leiden_cols))
    leiden_palette = {k: v for k, v in bgs4_leiden_palette.items() if k in set(leiden_clusters)}

# Lock viewbox (same for every panel)
X = harmonized.obsm["X_umap"]
pad = 0.02
xmin, xmax = X[:, 0].min(), X[:, 0].max()
ymin, ymax = X[:, 1].min(), X[:, 1].max()
xr, yr = (xmax - xmin), (ymax - ymin)
xlim = (xmin - pad * xr, xmax + pad * xr)
ylim = (ymin - pad * yr, ymax + pad * yr)

# Grid sizing
n = len(leiden_clusters)
ncols = 6
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(ncols * 2.4, nrows * 2.4),
    squeeze=False
)

# Plot each leiden cluster in its own panel
# - Background CosMx (other clusters): grey
# - Background snm3C: red
# - Foreground CosMx for this cluster: colored by leiden palette
for i, cl in enumerate(leiden_clusters):
    r, c = divmod(i, ncols)
    ax = axes[r, c]

    fg_mask = cosmx_mask & (harmonized.obs[leiden_col].astype(str) == str(cl))

    # 1) Background CosMx (excluding this cluster) in grey
    bg_cosmx_mask = cosmx_mask & (~fg_mask)
    sc.pl.umap(
        harmonized[bg_cosmx_mask].copy(),
        color=None,
        size=0.02,
        ax=ax,
        show=False
    )
    bg1 = ax.collections[-1]
    bg1.set_facecolor((0.80, 0.80, 0.80, 1.0))
    bg1.set_edgecolor((0.80, 0.80, 0.80, 1.0))
    bg1.set_alpha(0.18)

    # 2) Background snm3C in red (uniform)
    sc.pl.umap(
        harmonized[m3c_mask].copy(),
        color=None,
        size=0.02,
        ax=ax,
        show=False
    )
    bg2 = ax.collections[-1]
    bg2.set_facecolor('#D62728')  # red
    bg2.set_edgecolor('#D62728')
    bg2.set_alpha(0.25)

    # 3) Foreground: CosMx cells of this leiden cluster (colored)
    fg = harmonized[fg_mask].copy()
    fg.obs[leiden_col] = fg.obs[leiden_col].astype('category')
    fg.obs[leiden_col] = fg.obs[leiden_col].cat.remove_unused_categories()

    cl_color = leiden_palette.get(cl, None)
    if cl_color is None:
        sc.pl.umap(
            fg,
            color=leiden_col,
            size=2.6,
            ax=ax,
            show=False,
            legend_loc=None
        )
    else:
        sc.pl.umap(
            fg,
            color=leiden_col,
            palette={cl: cl_color},
            size=2.8,
            ax=ax,
            show=False,
            legend_loc=None
        )

    # same view + aspect everywhere
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")

    ax.set_title(f"leiden {cl}")
    ax.set_axis_off()
    for spine in ax.spines.values():
        spine.set_visible(False)

# Turn off empty panels
for j in range(n, nrows * ncols):
    r, c = divmod(j, ncols)
    axes[r, c].axis("off")

plt.tight_layout()
plt.show()

#cosmx cells only leidens in joint umap (m3C in background in red) --> there are no cosmx cells in that top right

In [ ]:
import pandas as pd

label_col = "adjusted_L3"   # your L3_adjusted cell type column (named adjusted_L3 in this object)
astro_mask = snm3_2t.obs[label_col] == "Astro"

astro_obs = snm3_2t.obs.loc[astro_mask].copy()

print(f"Total Astro cells: {astro_obs.shape[0]:,}")
print()

# 1) Age breakdowns
for col in ["age_groups", "fine_age_groups", "fine2_age_groups", "float_age", "age"]:
    if col in astro_obs.columns:
        print(f"=== {col} ===")
        if astro_obs[col].dtype.name in ["category", "object"]:
            print(astro_obs[col].value_counts(dropna=False).head(50))
        else:
            print(astro_obs[col].describe())
        print()

# 2) Sample / donor / region breakdowns
for col in ["sample", "Case", "ROI", "Section", "updated_region", "region", "merge_region", "sequenceBatch", "batch"]:
    if col in astro_obs.columns:
        print(f"=== {col} ===")
        print(astro_obs[col].value_counts(dropna=False).head(50))
        print()

# 3) Existing subtype annotations (if present)
for col in ["L2", "L3", "sL3", "newL4", "SubType", "MajorType"]:
    if col in astro_obs.columns:
        print(f"=== {col} ===")
        print(astro_obs[col].value_counts(dropna=False).head(50))
        print()

# 4) Useful joint breakdown tables (top combos)
def top_pairs(df, a, b, n=30):
    if a in df.columns and b in df.columns:
        print(f"=== Top {n} combos: {a} × {b} ===")
        print(df.groupby([a, b]).size().sort_values(ascending=False).head(n))
        print()

top_pairs(astro_obs, "age_groups", "sample", n=30)
top_pairs(astro_obs, "age_groups", "updated_region", n=30)
top_pairs(astro_obs, "sample", "updated_region", n=30)
top_pairs(astro_obs, "SubType", "updated_region", n=30)

# 5) Quick sanity: how many unique values exist per key column?
cols_to_check = [
    "age_groups", "fine_age_groups", "fine2_age_groups",
    "sample", "Case", "ROI", "Section",
    "updated_region", "region", "merge_region",
    "sequenceBatch", "batch",
    "L2", "L3", "sL3", "newL4", "MajorType", "SubType"
]
print("=== Unique counts (Astro only) ===")
for col in cols_to_check:
    if col in astro_obs.columns:
        print(f"{col:15s}: {astro_obs[col].nunique(dropna=False)}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc
import math

# Style
mpl.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
})

# Helper: lock UMAP viewbox
X = harmonized.obsm["X_umap"]
pad = 0.02
xmin, xmax = X[:, 0].min(), X[:, 0].max()
ymin, ymax = X[:, 1].min(), X[:, 1].max()
xr, yr = (xmax - xmin), (ymax - ymin)
xlim = (xmin - pad * xr, xmax + pad * xr)
ylim = (ymin - pad * yr, ymax + pad * yr)

# Identify snm3_2t-derived cells inside harmonized
# We'll use sample overlap by default
snm3_2t_samples = set(snm3_2t.obs["sample"].dropna().astype(str).unique())
snm3_2t_cases   = set(snm3_2t.obs["Case"].dropna().astype(str).unique())

harm_sample = harmonized.obs["sample"].astype(str) if "sample" in harmonized.obs.columns else None
harm_case   = harmonized.obs["Case"].astype(str) if "Case" in harmonized.obs.columns else None

# Prefer sample matching (most specific)
if harm_sample is not None:
    in_snm3_2t = harm_sample.isin(snm3_2t_samples)
elif harm_case is not None:
    in_snm3_2t = harm_case.isin(snm3_2t_cases)
else:
    raise ValueError("harmonized.obs must contain 'sample' or 'Case' to locate snm3_2t cells.")

print(f"snm3_2t-like cells found in harmonized: {in_snm3_2t.sum():,} / {harmonized.n_obs:,}")

# Generic grid plotter
def plot_grid_by_group(group_col, title, ncols=6, max_groups=36):
    if group_col not in harmonized.obs.columns:
        print(f"Skipping {group_col}: not found in harmonized.obs")
        return

    # groups present within snm3_2t-like cells only
    groups = (
        harmonized.obs.loc[in_snm3_2t, group_col]
        .dropna()
        .astype(str)
        .value_counts()
    )

    # keep top groups only (avoid huge grids)
    groups = groups.head(max_groups)

    group_names = list(groups.index)
    n = len(group_names)
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(ncols * 2.2, nrows * 2.2),
        squeeze=False
    )

    for i, g in enumerate(group_names):
        r, c = divmod(i, ncols)
        ax = axes[r, c]

        fg_mask = in_snm3_2t & (harmonized.obs[group_col].astype(str) == g)
        bg_mask = ~fg_mask

        # Background (all other cells) grey
        sc.pl.umap(
            harmonized[bg_mask].copy(),
            color=None,
            size=0.05,
            ax=ax,
            show=False
        )
        bg = ax.collections[-1]
        bg.set_facecolor((0.80, 0.80, 0.80, 1.0))
        bg.set_edgecolor((0.80, 0.80, 0.80, 1.0))
        bg.set_alpha(0.12)

        # Foreground (this group) red
        sc.pl.umap(
            harmonized[fg_mask].copy(),
            color=None,
            size=0.25,
            ax=ax,
            show=False
        )
        fg = ax.collections[-1]
        fg.set_facecolor("#D62728")
        fg.set_edgecolor("#D62728")
        fg.set_alpha(0.85)

        # consistent view
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.set_aspect("equal", adjustable="box")

        ax.set_title(f"{g}\n(n={fg_mask.sum():,})")
        ax.set_axis_off()

    # turn off unused panels
    for j in range(n, nrows * ncols):
        r, c = divmod(j, ncols)
        axes[r, c].axis("off")

    fig.suptitle(title, y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

# 1) Grid by age
plot_grid_by_group(
    group_col="age",
    title="Where snm3_2t cells go in harmonized UMAP - split by age",
    ncols=4,
    max_groups=12
)

# 2) Grid by sample
plot_grid_by_group(
    group_col="sample",
    title="Where snm3_2t cells go in harmonized UMAP - split by sample",
    ncols=6,
    max_groups=36
)

# 3) Grid by Case
plot_grid_by_group(
    group_col="Case",
    title="Where snm3_2t cells go in harmonized UMAP - split by Case",
    ncols=5,
    max_groups=25
)

In [ ]:
# If you have integrated object with harmony correction
knn_label_transfer_from_harmony(
    integrated_adata=harmonized_2t,  # your harmony-corrected concatenated object
    target_adata=bgs4_lge_mge,  # your original CosMx spatial data
    label_key='adjusted_L3',
    batch_key='batch',
    target_batch='0',  # whatever identifier you used
    harmony_key='X_pca_harmony',
    n_neighbors=20,  # adjust based on your data
    confidence_threshold=0.6       ##### back to the usual
)

# Now your spatial object has transferred labels you can visualize
sc.pl.umap(bgs4_lge_mge, color='adjusted_L3_transferred')

In [ ]:
#change bgs4 '?' cell type to 'eMGE'

label_col = 'adjusted_L3_transferred'

bgs4_lge_mge.obs[label_col] = (
    bgs4_lge_mge.obs[label_col]
    .replace({'?': 'eMGE'})
)

In [ ]:
# If you have integrated object with harmony correction
knn_label_transfer_from_harmony(
    integrated_adata=harmonized_2t,  # your harmony-corrected concatenated object
    target_adata=bgs4_lge_mge,  # your original CosMx spatial data
    label_key='adjusted_L3',
    batch_key='batch',
    target_batch='0',  # whatever identifier you used
    harmony_key='X_pca_harmony',
    n_neighbors=20,  # adjust based on your data
    confidence_threshold=0.55       #########
)

# Now your spatial object has transferred labels you can visualize
sc.pl.umap(bgs4_lge_mge, color='adjusted_L3_transferred')

In [ ]:
# If you have integrated object with harmony correction
knn_label_transfer_from_harmony(
    integrated_adata=harmonized_2t,  # your harmony-corrected concatenated object
    target_adata=bgs4_lge_mge,  # your original CosMx spatial data
    label_key='adjusted_L3',
    batch_key='batch',
    target_batch='0',  # whatever identifier you used
    harmony_key='X_pca_harmony',
    n_neighbors=20,  # adjust based on your data
    confidence_threshold=0.5       #########
)

# Now your spatial object has transferred labels you can visualize
sc.pl.umap(bgs4_lge_mge, color='adjusted_L3_transferred')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib as mpl
import scanpy as sc

# GLOBAL FONT SETTINGS (apply everywhere)
mpl.rcParams.update({
    'font.size': 18,          # base font
    'axes.titlesize': 20,     # plot title
    'axes.labelsize': 22,     # axis labels
    'xtick.labelsize': 18,    # tick labels
    'ytick.labelsize': 18,
    'legend.fontsize': 18,    # legend labels
    'legend.title_fontsize': 18
})

# Rename batch categories
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

# Per-cell sizes (CosMx smaller, snm3C larger)
sizes = np.where(
    harmonized.obs['batch_named'] == 'CosMx',
    0.7,   # CosMx (dense  smaller)
    1.2     # snm3C (sparser  larger)
)

# Plot UMAP WITHOUT default legend
sc.pl.umap(
    harmonized,
    color='batch_named',
    size=sizes,
    palette={
        'CosMx': '#5DA5DA',
        'snm3C': '#D62728'
    },
    title="3T m3C  GW35 CosMx joint embedding",
    legend_loc=None,   # suppress Scanpy dot legend
    show=False
)

ax = plt.gca()

# Custom square-color legend
legend_handles = [
    mpatches.Patch(color='#5DA5DA', label='CosMx'),
    mpatches.Patch(color='#D62728', label='snm3C-seq'),
]

ax.legend(
    handles=legend_handles,
    #title="Batch",
    loc='lower left',
    frameon=False,
    fontsize=13,
    title_fontsize=18
)

# Final polish
ax.set_title("2301+2303 - 2T snm3C and GW24 CosMx joint embedding", fontsize=16)

# remove frame / box around UMAP
ax.set_axis_off()          # removes spines + ticks + axis labels
# (optional) also ensure spines are off
for spine in ax.spines.values():
    spine.set_visible(False)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches

# CosMx-only JOINT UMAP (coords from harmonized)
# - Colored by bgs4_lge_mge transferred cell types
# - ONLY the specified cell types, in the specified order
# - Custom explicit colors
# - Arial + editable text in Illustrator (PDF)
# - Rasterized points (small PDF)
# - Legend OUTSIDE, very close, with RECTANGULAR swatches
# - Hide axis tick numbers
# - Bigger title
# - REMOVE ALL SPINES / FRAME

# Objects
adata = bgs4_lge_mge
harm = harmonized

# Required keys / settings
color_key = 'adjusted_L3_transferred'  # labels live in bgs4_lge_mge
batch_key = 'batch'                    # harmonized batch column
target_batch = '0'                     # CosMx inside harmonized
id_col = 'cell_id'                     # fallback matching key if obs_names differ

# Only keep these cell types + colors
cell_type_order = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    'eMGE',
    'GPC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

# Explicit colors
cell_type_colors = {
    'eMGE': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
}

# Aesthetics
umap_s = 0.3
alpha = 1
title_txt = "2T snm3C and GW24 CosMx joint embedding"

# GLOBAL FONT + PDF SETTINGS (Arial + editable text in Illustrator)
mpl.rcParams.update({
    # Arial everywhere
    'font.family': 'Arial',
    'font.sans-serif': ['Arial'],

    'font.size': 18,
    'axes.titlesize': 30,
    'axes.labelsize': 14,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 12,
    'legend.title_fontsize': 14,

    # Keep text editable in PDF
    'pdf.fonttype': 42,
    'ps.fonttype': 42,

    # If you ever save SVG too
    'svg.fonttype': 'none',
})

# Sanity checks
if 'X_umap' not in harm.obsm:
    raise ValueError("harmonized.obsm['X_umap'] not found.")
if color_key not in adata.obs.columns:
    raise KeyError(f"{color_key} not found in bgs4_lge_mge.obs")
if batch_key not in harm.obs.columns:
    raise KeyError(f"{batch_key} not found in harmonized.obs")

# Restrict harmonized to CosMx batch (UMAP coords for CosMx only)
harm_cosmx_mask = harm.obs[batch_key].astype(str).values == str(target_batch)
harm_umap_cosmx = harm.obsm['X_umap'][harm_cosmx_mask, :]
harm_cosmx_names = harm.obs_names[harm_cosmx_mask]

# Align harmonized CosMx UMAP rows to bgs4_lge_mge cells
if np.array_equal(harm_cosmx_names.values, adata.obs_names.values):
    umap_coords = harm_umap_cosmx
else:
    if id_col not in adata.obs.columns or id_col not in harm.obs.columns:
        raise ValueError(
            "CosMx cells in harmonized do not match bgs4_lge_mge.obs_names, and "
            f"'{id_col}' is missing from obs in one of the objects. "
            "Either ensure obs_names match OR set id_col to a shared unique ID."
        )

    harm_cosmx_obs = harm.obs.loc[harm_cosmx_mask, [id_col]].copy()
    harm_cosmx_obs['__umap_i'] = np.arange(harm_cosmx_obs.shape[0])

    bgs4_lge_mge_obs = adata.obs[[id_col]].copy()
    bgs4_lge_mge_obs['__bgs4_lge_mge_i'] = np.arange(bgs4_lge_mge_obs.shape[0])

    merged = bgs4_lge_mge_obs.merge(harm_cosmx_obs, on=id_col, how='inner')

    if merged.shape[0] == 0:
        raise ValueError(
            f"No cells matched between bgs4_lge_mge and harmonized CosMx cells using id_col='{id_col}'."
        )

    if merged.shape[0] != adata.n_obs:
        print(
            f"Warning: matched {merged.shape[0]:,} / {adata.n_obs:,} bgs4_lge_mge cells "
            f"to harmonized CosMx cells via '{id_col}'. Proceeding with matched subset only."
        )

    umap_coords = harm_umap_cosmx[merged['__umap_i'].values, :]
    adata = adata[merged['__bgs4_lge_mge_i'].values].copy()

# Keep ONLY specified cell types
labels_all = adata.obs[color_key].astype(str)
mask_keep = labels_all.isin(set(cell_type_order))

adata = adata[mask_keep].copy()
umap_coords = umap_coords[mask_keep.values, :]
labels = adata.obs[color_key].astype(str)

present = [ct for ct in cell_type_order if ct in set(labels.unique())]
label_colors = {ct: cell_type_colors.get(ct, 'lightgrey') for ct in present}
point_colors = [label_colors.get(l, 'lightgrey') for l in labels.values]

# RANDOMIZE POINT DRAW ORDER
rng = np.random.default_rng(0)
perm = rng.permutation(umap_coords.shape[0])
umap_coords = umap_coords[perm]
point_colors = np.array(point_colors)[perm]

# Plot
fig, ax = plt.subplots(figsize=(9, 5))

scat = ax.scatter(
    umap_coords[:, 0], umap_coords[:, 1],
    c=point_colors,
    s=umap_s,
    alpha=alpha,
    linewidths=0
)

# Rasterize points (keeps PDF small)
scat.set_rasterized(True)

ax.set_title(title_txt, fontsize=17)
# ax.set_xlabel("")
# ax.set_ylabel("")

# Hide tick numbers (keep clean axes)
ax.set_xticks([])
ax.set_yticks([])
ax.tick_params(bottom=False, left=False)

# REMOVE ALL SPINES / FRAME
for spine in ax.spines.values():
    spine.set_visible(False)

# Legend (outside, very close)
legend_handles = [
    mpatches.Patch(facecolor=label_colors[ct], edgecolor='none', label=ct)
    for ct in present
]

ax.legend(
    handles=legend_handles,
    loc='center left',
    bbox_to_anchor=(1.005, 0.5),
    frameon=False,
    borderaxespad=0.0,
    handlelength=1.2,
    handleheight=0.9,
)

plt.tight_layout(rect=[0, 0, 0.86, 1])

# SAVE AS PDF (Illustrator-ready: editable text + small file)
# out_pdf = "GW24_CosMx_joint_embedding_celltypes_filtered.pdf"
# plt.savefig(
#     out_pdf,
#     format="pdf",
#     bbox_inches="tight",
#     dpi=300,         # raster layer resolution; drop to 200 for smaller
#     transparent=True
# )

plt.show()
# print(f"Saved: {out_pdf}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# Grid: 2 cell types per row
# For each cell type:
#   LEFT  = spatial (all other types grey)
#   RIGHT = joint UMAP (CosMx-only coords from harmonized; all other types grey)
# Only the 13 "important" cell types are shown.

# Objects / keys
adata_full = bgs4_lge_mge
harm = harmonized

categ = 'adjusted_L3_transferred'   # cell type labels (in adata)
batch_key = 'batch'                 # harmonized batch column
target_batch = '0'                  # CosMx inside harmonized
id_col = 'cell_id'                  # fallback matching key if obs_names differ

# The 13 important cell types (order)
cell_type_order = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    'eMGE',
    'GPC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

# Colors (same as your joint UMAP)
cell_type_colors = {
    'eMGE': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
}

# Style (Illustrator-friendly text)
mpl.rcParams.update({
    'font.family': 'Arial',
    'font.sans-serif': ['Arial'],
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
    'font.size': 12,
})

# Aesthetics
grey_color = '#D3D3D3'
bg_alpha_spatial = 0.25
fg_alpha_spatial = 1.0
bg_alpha_umap = 0.15
fg_alpha_umap = 1.0

spot_size_spatial = 0.5
spot_size_umap = 0.3

# Sanity checks
if 'spatial' not in adata_full.obsm:
    raise ValueError("bgs4_lge_mge.obsm['spatial'] not found.")
if categ not in adata_full.obs.columns:
    raise KeyError(f"{categ} not found in bgs4_lge_mge.obs")
if 'X_umap' not in harm.obsm:
    raise ValueError("harmonized.obsm['X_umap'] not found.")
if batch_key not in harm.obs.columns:
    raise KeyError(f"{batch_key} not found in harmonized.obs")

# Build CosMx-only UMAP coords aligned to adata_full
harm_cosmx_mask = harm.obs[batch_key].astype(str).values == str(target_batch)
harm_umap_cosmx = harm.obsm['X_umap'][harm_cosmx_mask, :]
harm_cosmx_names = harm.obs_names[harm_cosmx_mask]

# Default: assume same ordering/obs_names
adata_aligned = adata_full
umap_coords_aligned = None

if np.array_equal(harm_cosmx_names.values, adata_full.obs_names.values):
    umap_coords_aligned = harm_umap_cosmx
else:
    # fallback: align using shared id_col
    if (id_col not in adata_full.obs.columns) or (id_col not in harm.obs.columns):
        raise ValueError(
            "CosMx cells in harmonized do not match bgs4_lge_mge.obs_names, and "
            f"'{id_col}' is missing from obs in one of the objects. "
            "Either ensure obs_names match OR set id_col to a shared unique ID."
        )

    harm_cosmx_obs = harm.obs.loc[harm_cosmx_mask, [id_col]].copy()
    harm_cosmx_obs['__umap_i'] = np.arange(harm_cosmx_obs.shape[0])

    ad_obs = adata_full.obs[[id_col]].copy()
    ad_obs['__adata_i'] = np.arange(ad_obs.shape[0])

    merged = ad_obs.merge(harm_cosmx_obs, on=id_col, how='inner')

    if merged.shape[0] == 0:
        raise ValueError(
            f"No cells matched between bgs4_lge_mge and harmonized CosMx cells using id_col='{id_col}'."
        )

    if merged.shape[0] != adata_full.n_obs:
        print(
            f"Warning: matched {merged.shape[0]:,} / {adata_full.n_obs:,} bgs4_lge_mge cells "
            f"to harmonized CosMx cells via '{id_col}'. Proceeding with matched subset only."
        )

    umap_coords_aligned = harm_umap_cosmx[merged['__umap_i'].values, :]
    adata_aligned = adata_full[merged['__adata_i'].values].copy()

# Restrict to the 13 cell types (for BOTH spatial and UMAP)
labels_all = adata_aligned.obs[categ].astype(str)
mask_keep = labels_all.isin(set(cell_type_order))

adata = adata_aligned[mask_keep].copy()
umap_coords = umap_coords_aligned[mask_keep.values, :]
labels = adata.obs[categ].astype(str).values

spatial_coords = adata.obsm['spatial']
present = [ct for ct in cell_type_order if ct in set(labels)]

# Plot grid: 2 cell types per row, each cell type = (spatial, umap)
# => 4 columns total per row: [spatial ct1, umap ct1, spatial ct2, umap ct2]
n_types = len(present)
nrows = int(np.ceil(n_types / 2))
ncols = 4

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(4.8 * ncols, 4.8 * nrows),
    squeeze=False
)

for i, ct in enumerate(present):
    row = i // 2
    slot = i % 2  # 0 or 1 within the row
    ax_sp = axes[row, slot * 2]
    ax_um = axes[row, slot * 2 + 1]

    ct_color = cell_type_colors.get(ct, 'lightgrey')
    ct_mask = (labels == ct)

    # LEFT: Spatial
    # background (other cell types)
    ax_sp.scatter(
        spatial_coords[~ct_mask, 0],
        spatial_coords[~ct_mask, 1],
        c=grey_color,
        s=spot_size_spatial,
        alpha=bg_alpha_spatial,
        linewidths=0
    )
    # foreground (this cell type)
    ax_sp.scatter(
        spatial_coords[ct_mask, 0],
        spatial_coords[ct_mask, 1],
        c=ct_color,
        s=spot_size_spatial,
        alpha=fg_alpha_spatial,
        linewidths=0
    )

    ax_sp.set_title(f"{ct} - spatial", fontsize=12)
    ax_sp.set_aspect('equal')
    ax_sp.invert_yaxis()
    ax_sp.set_xticks([])
    ax_sp.set_yticks([])
    ax_sp.tick_params(bottom=False, left=False)
    for spine in ax_sp.spines.values():
        spine.set_visible(False)

    # RIGHT: UMAP
    # background (other cell types)
    ax_um.scatter(
        umap_coords[~ct_mask, 0],
        umap_coords[~ct_mask, 1],
        c=grey_color,
        s=spot_size_umap,
        alpha=bg_alpha_umap,
        linewidths=0
    )
    # foreground (this cell type)
    scat_fg = ax_um.scatter(
        umap_coords[ct_mask, 0],
        umap_coords[ct_mask, 1],
        c=ct_color,
        s=spot_size_umap,
        alpha=fg_alpha_umap,
        linewidths=0
    )
    # keep PDFs small
    scat_fg.set_rasterized(True)

    ax_um.set_title(f"{ct} - joint UMAP", fontsize=12)
    ax_um.set_xticks([])
    ax_um.set_yticks([])
    ax_um.tick_params(bottom=False, left=False)
    for spine in ax_um.spines.values():
        spine.set_visible(False)

# Turn off any unused axes in the last row (if odd number of types)
if n_types % 2 == 1:
    axes[-1, 2].axis('off')
    axes[-1, 3].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import anndata as ad

# 1) Subset to 0 vars (this changes shape to (n_obs, 0))
harm_slim = harmonized_2t[:, []].copy()

# 2) Drop big matrices / layers / raw
harm_slim.X = None
harm_slim.layers.clear()
harm_slim.raw = None
harm_slim.obsp.clear()
harm_slim.varm.clear()
harm_slim.uns = {}

# 3) Keep only UMAP embedding (and optionally other embeddings you want)
harm_slim.obsm = {'X_umap': harmonized.obsm['X_umap'].copy()}

# 4) Write
harm_slim.write("20260118_bgs4_2T_2301+3_harmonized_joint_umap_slim.h5ad", compression="gzip")

#moved to processed_data/20260106

In [ ]:
#resegmented

bgs4 = sc.read_h5ad("/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20260106/20260107_bgs4_reseg_snm3_2T_L3.h5ad")

bgs4

In [ ]:
#earlier today saved the harmonized object umap only

harmonized = sc.read_h5ad("/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20260106/20260114_bgs4_2T_harmonized_joint_umap_slim.h5ad")
harmonized

In [ ]:
#rename '?' to 'eMGE'

label_col = 'adjusted_L3_transferred'

bgs4.obs[label_col] = (
    bgs4.obs[label_col]
    .replace({'?': 'eMGE'})
)

In [ ]:
# Rename batch categories for plotting
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

sc.pl.umap(
    harmonized,
    color='batch_named',
    size=1.5,
    palette={
    'CosMx': '#5DA5DA',
    'snm3C': '#D62728'
    }
)

#contrast

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Rename batch categories
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

# Per-cell sizes
sizes = np.where(
    harmonized.obs['batch_named'] == 'CosMx',
    0.35,   # smaller CosMx
    1.4    # larger snm3C
)

# Plot WITHOUT legend
sc.pl.umap(
    harmonized,
    color='batch_named',
    size=sizes,
    palette={
        'CosMx': '#5DA5DA',
        'snm3C': '#D62728'
    },
    title="3T m3C  GW35 CosMx joint embedding",
    legend_loc=None,   # <-- suppress default dot legend
    show=False
)

ax = plt.gca()

# Build square legend handles
legend_handles = [
    mpatches.Patch(color='#5DA5DA', label='CosMx'),
    mpatches.Patch(color='#D62728', label='snm3C'),
]

ax.legend(
    handles=legend_handles,
    title="Batch",
    loc='best',
    frameon=False
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib as mpl
import scanpy as sc

# GLOBAL FONT SETTINGS (apply everywhere)
mpl.rcParams.update({
    'font.size': 18,          # base font
    'axes.titlesize': 24,     # plot title
    'axes.labelsize': 22,     # axis labels
    'xtick.labelsize': 18,    # tick labels
    'ytick.labelsize': 18,
    'legend.fontsize': 18,    # legend labels
    'legend.title_fontsize': 18
})

# Rename batch categories
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

# Per-cell sizes (CosMx smaller, snm3C larger)
sizes = np.where(
    harmonized.obs['batch_named'] == 'CosMx',
    0.7,   # CosMx (dense  smaller)
    1.2     # snm3C (sparser  larger)
)

# Plot UMAP WITHOUT default legend
sc.pl.umap(
    harmonized,
    color='batch_named',
    size=sizes,
    palette={
        'CosMx': '#5DA5DA',
        'snm3C': '#D62728'
    },
    title="3T m3C  GW35 CosMx joint embedding",
    legend_loc=None,   # suppress Scanpy dot legend
    show=False
)

ax = plt.gca()

# Custom square-color legend
legend_handles = [
    mpatches.Patch(color='#5DA5DA', label='CosMx'),
    mpatches.Patch(color='#D62728', label='snm3C-seq'),
]

ax.legend(
    handles=legend_handles,
    #title="Batch",
    loc='lower left',
    frameon=False,
    fontsize=13,
    title_fontsize=18
)

# Final polish
ax.set_title("2T snm3C and GW24 CosMx joint embedding", fontsize=16)

# remove frame / box around UMAP
ax.set_axis_off()          # removes spines + ticks + axis labels
# (optional) also ensure spines are off
for spine in ax.spines.values():
    spine.set_visible(False)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib as mpl
import scanpy as sc

# GLOBAL FONT + SVG SETTINGS (Illustrator-friendly)
mpl.rcParams.update({
    'font.size': 18,          # base font
    'axes.titlesize': 24,     # plot title
    'axes.labelsize': 22,     # axis labels
    'xtick.labelsize': 18,    # tick labels
    'ytick.labelsize': 18,
    'legend.fontsize': 18,    # legend labels
    'legend.title_fontsize': 18,
    'svg.fonttype': 'none',   # keep text editable in SVG
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})

# Rename batch categories
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

# Per-cell sizes (CosMx smaller, snm3C larger)
sizes = np.where(
    harmonized.obs['batch_named'] == 'CosMx',
    0.7,   # CosMx (dense  smaller)
    1.2    # snm3C (sparser  larger)
)

# Plot UMAP WITHOUT default legend
sc.pl.umap(
    harmonized,
    color='batch_named',
    size=sizes,
    palette={
        'CosMx': '#5DA5DA',
        'snm3C': '#D62728'
    },
    title="3T m3C  GW35 CosMx joint embedding",
    legend_loc=None,   # suppress Scanpy legend
    show=False
)

ax = plt.gca()

# Rasterize the UMAP points (keeps SVG small/fast)
for coll in ax.collections:
    coll.set_rasterized(True)

# Custom square-color legend
legend_handles = [
    mpatches.Patch(color='#5DA5DA', label='CosMx'),
    mpatches.Patch(color='#D62728', label='snm3C'),
]

ax.legend(
    handles=legend_handles,
    loc='lower left',
    frameon=False,
    fontsize=15,
    title_fontsize=18
)

# Final polish
ax.set_title("2T snm3C and GW24 CosMx joint embedding", fontsize=16)
ax.tick_params(axis='both', which='major', labelsize=6)

# SAVE AS SVG (Illustrator-ready, rasterized points)
plt.savefig(
    "2T_snm3C_GW24_CosMx_joint_embedding_batch.svg",
    bbox_inches="tight",
    dpi=600  # controls raster resolution of points inside the SVG
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib as mpl
import scanpy as sc

# GLOBAL FONT + PDF SETTINGS (Arial + editable text in Illustrator)
mpl.rcParams.update({
    # Font
    'font.family': 'Arial',
    'font.sans-serif': ['Arial'],

    # Sizes
    'font.size': 18,
    'axes.titlesize': 24,
    'axes.labelsize': 22,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 18,
    'legend.title_fontsize': 18,

    # Keep text as text (editable) in PDF
    'pdf.fonttype': 42,
    'ps.fonttype': 42,

    # If you ever save SVG too
    'svg.fonttype': 'none',
})

# Rename batch categories
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

# Per-cell sizes (CosMx smaller, snm3C larger)
sizes = np.where(
    harmonized.obs['batch_named'] == 'CosMx',
    0.7,   # CosMx (dense  smaller)
    1.2    # snm3C (sparser  larger)
)

# Plot UMAP WITHOUT default legend
sc.pl.umap(
    harmonized,
    color='batch_named',
    size=sizes,
    palette={
        'CosMx': '#5DA5DA',
        'snm3C': '#D62728'
    },
    title="3T m3C  GW35 CosMx joint embedding",
    legend_loc=None,
    show=False
)

ax = plt.gca()

# Rasterize ONLY the point cloud (small PDF), keep text/legend vector
for coll in ax.collections:
    coll.set_rasterized(True)

# Custom square-color legend (stays vector)
legend_handles = [
    mpatches.Patch(color='#5DA5DA', label='CosMx'),
    mpatches.Patch(color='#D62728', label='snm3C-seq'),
]

ax.legend(
    handles=legend_handles,
    loc='lower left',
    frameon=False,
    fontsize=13,
    title_fontsize=18
)

# Final polish
ax.set_title("2T snm3C and GW24 CosMx joint embedding", fontsize=16)

# remove frame / box around UMAP
ax.set_axis_off()
for spine in ax.spines.values():
    spine.set_visible(False)

# SAVE AS PDF (small file; dots rasterized; text stays editable)
out_pdf = "2T_GW24_umap_joint_batch_20260115.pdf"
plt.savefig(
    out_pdf,
    format="pdf",
    bbox_inches="tight",
    dpi=300,         # controls dot raster resolution; try 200 for even smaller
    transparent=True
)

plt.show()
print(f"Saved: {out_pdf}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib as mpl
import scanpy as sc

# GLOBAL FONT + PDF SETTINGS (Arial + editable text in Illustrator)
mpl.rcParams.update({
    # Font
    'font.family': 'Arial',
    'font.sans-serif': ['Arial'],

    # Sizes
    'font.size': 18,
    'axes.titlesize': 24,
    'axes.labelsize': 22,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 18,
    'legend.title_fontsize': 18,

    # Keep text as text (editable) in PDF
    'pdf.fonttype': 42,
    'ps.fonttype': 42,

    # If you ever save SVG too
    'svg.fonttype': 'none',
})

# Rename batch categories
harmonized.obs['batch_named'] = harmonized.obs['batch'].map({
    '0': 'CosMx',
    '1': 'snm3C',
    0: 'CosMx',
    1: 'snm3C'
}).astype('category')

# Per-cell sizes (CosMx smaller, snm3C larger)
sizes = np.where(
    harmonized.obs['batch_named'] == 'CosMx',
    0.7,   # CosMx (dense  smaller)
    1.2    # snm3C (sparser  larger)
)

# Plot UMAP WITHOUT default legend
sc.pl.umap(
    harmonized,
    color='batch_named',
    size=sizes,
    palette={
        'CosMx': '#5DA5DA',
        'snm3C': '#D62728'
    },
    title="3T m3C  GW35 CosMx joint embedding",
    legend_loc=None,
    show=False
)

ax = plt.gca()

# Rasterize ONLY the point cloud (small PDF), keep text/legend vector
for coll in ax.collections:
    coll.set_rasterized(True)

# Custom square-color legend (stays vector)
legend_handles = [
    mpatches.Patch(color='#5DA5DA', label='CosMx'),
    mpatches.Patch(color='#D62728', label='snm3C-seq'),
]

ax.legend(
    handles=legend_handles,
    loc='lower left',
    frameon=False,
    fontsize=13,
    title_fontsize=18
)

# Final polish
ax.set_title("2T snm3C and GW24 CosMx joint embedding", fontsize=16)

# remove frame / box around UMAP
ax.set_axis_off()
for spine in ax.spines.values():
    spine.set_visible(False)

# SAVE AS PDF (small file; dots rasterized; text stays editable)
# out_pdf = "2T_GW24_umap_joint_batch_20260115.pdf"
# plt.savefig(
#     out_pdf,
#     format="pdf",
#     bbox_inches="tight",
#     dpi=300,         # controls dot raster resolution; try 200 for even smaller
#     transparent=True
# )

plt.show()
#print(f"Saved: {out_pdf}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches

# CosMx-only JOINT UMAP (coords from harmonized)
# - Colored by bgs4 transferred cell types
# - ONLY the specified cell types, in the specified order
# - Custom explicit colors
# - Illustrator-friendly SVG (editable text)
# - Rasterized points (small/fast SVG)
# - Legend OUTSIDE, very close, with RECTANGULAR swatches
# - Hide axis tick numbers
# - Bigger title

# Objects
adata = bgs4
harm = harmonized

# Required keys / settings
color_key = 'adjusted_L3_transferred'  # labels live in bgs4
batch_key = 'batch'                    # harmonized batch column
target_batch = '0'                     # CosMx inside harmonized
id_col = 'cell_id'                     # fallback matching key if obs_names differ

# Only keep these cell types + colors
cell_type_order = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    'eMGE',
    'GPC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

# Explicit colors
cell_type_colors = {
    'eMGE': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
}

# Aesthetics
umap_s = 0.3
alpha = 1
title_txt = "2T snm3C and GW24 CosMx joint embedding"

# GLOBAL FONT + SVG SETTINGS (Illustrator-friendly)
mpl.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 30,
    'axes.labelsize': 14,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 12,
    'legend.title_fontsize': 14,
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})

# Sanity checks
if 'X_umap' not in harm.obsm:
    raise ValueError("harmonized.obsm['X_umap'] not found.")
if color_key not in adata.obs.columns:
    raise KeyError(f"{color_key} not found in bgs4.obs")
if batch_key not in harm.obs.columns:
    raise KeyError(f"{batch_key} not found in harmonized.obs")

# Restrict harmonized to CosMx batch (UMAP coords for CosMx only)
harm_cosmx_mask = harm.obs[batch_key].astype(str).values == str(target_batch)
harm_umap_cosmx = harm.obsm['X_umap'][harm_cosmx_mask, :]
harm_cosmx_names = harm.obs_names[harm_cosmx_mask]

# Align harmonized CosMx UMAP rows to bgs4 cells
if np.array_equal(harm_cosmx_names.values, adata.obs_names.values):
    umap_coords = harm_umap_cosmx
else:
    if id_col not in adata.obs.columns or id_col not in harm.obs.columns:
        raise ValueError(
            "CosMx cells in harmonized do not match bgs4.obs_names, and "
            f"'{id_col}' is missing from obs in one of the objects. "
            "Either ensure obs_names match OR set id_col to a shared unique ID."
        )

    harm_cosmx_obs = harm.obs.loc[harm_cosmx_mask, [id_col]].copy()
    harm_cosmx_obs['__umap_i'] = np.arange(harm_cosmx_obs.shape[0])

    bgs4_obs = adata.obs[[id_col]].copy()
    bgs4_obs['__bgs4_i'] = np.arange(bgs4_obs.shape[0])

    merged = bgs4_obs.merge(harm_cosmx_obs, on=id_col, how='inner')

    if merged.shape[0] == 0:
        raise ValueError(
            f"No cells matched between bgs4 and harmonized CosMx cells using id_col='{id_col}'."
        )

    if merged.shape[0] != adata.n_obs:
        print(
            f"Warning: matched {merged.shape[0]:,} / {adata.n_obs:,} bgs4 cells "
            f"to harmonized CosMx cells via '{id_col}'. Proceeding with matched subset only."
        )

    umap_coords = harm_umap_cosmx[merged['__umap_i'].values, :]
    adata = adata[merged['__bgs4_i'].values].copy()

# Keep ONLY specified cell types
labels_all = adata.obs[color_key].astype(str)
mask_keep = labels_all.isin(set(cell_type_order))

adata = adata[mask_keep].copy()
umap_coords = umap_coords[mask_keep.values, :]
labels = adata.obs[color_key].astype(str)

present = [ct for ct in cell_type_order if ct in set(labels.unique())]
label_colors = {ct: cell_type_colors.get(ct, 'lightgrey') for ct in present}
point_colors = [label_colors.get(l, 'lightgrey') for l in labels.values]

# RANDOMIZE POINT DRAW ORDER
rng = np.random.default_rng(0)
perm = rng.permutation(umap_coords.shape[0])
umap_coords = umap_coords[perm]
point_colors = np.array(point_colors)[perm]

# Plot
fig, ax = plt.subplots(figsize=(9, 5))

scat = ax.scatter(
    umap_coords[:, 0], umap_coords[:, 1],
    c=point_colors,
    s=umap_s,
    alpha=alpha,
    linewidths=0
)
scat.set_rasterized(True)

ax.set_title(title_txt, fontsize=17)
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_xticks([])
ax.set_yticks([])
ax.tick_params(bottom=False, left=False)

# Legend (outside, very close)
legend_handles = [
    mpatches.Patch(facecolor=label_colors[ct], edgecolor='none', label=ct)
    for ct in present
]

ax.legend(
    handles=legend_handles,
    loc='center left',
    bbox_to_anchor=(1.005, 0.5),
    frameon=False,
    borderaxespad=0.0,
    handlelength=1.2,
    handleheight=0.9,
)

plt.tight_layout(rect=[0, 0, 0.86, 1])

# SAVE AS SVG (Illustrator-ready, rasterized points)
plt.savefig(
    "GW24_CosMx_joint_embedding_celltypes_filtered.svg",
    bbox_inches="tight",
    dpi=600
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches

# CosMx-only JOINT UMAP (coords from harmonized)
# - Colored by bgs4 transferred cell types
# - ONLY the specified cell types, in the specified order
# - Custom explicit colors
# - Arial + editable text in Illustrator (PDF)
# - Rasterized points (small PDF)
# - Legend OUTSIDE, very close, with RECTANGULAR swatches
# - Hide axis tick numbers
# - Bigger title
# - REMOVE ALL SPINES / FRAME

# Objects
adata = bgs4
harm = harmonized

# Required keys / settings
color_key = 'adjusted_L3_transferred'  # labels live in bgs4
batch_key = 'batch'                    # harmonized batch column
target_batch = '0'                     # CosMx inside harmonized
id_col = 'cell_id'                     # fallback matching key if obs_names differ

# Only keep these cell types + colors
cell_type_order = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    'eMGE',
    'GPC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

# Explicit colors
cell_type_colors = {
    'eMGE': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
}

# Aesthetics
umap_s = 0.3
alpha = 1
title_txt = "2T snm3C and GW24 CosMx joint embedding"

# GLOBAL FONT + PDF SETTINGS (Arial + editable text in Illustrator)
mpl.rcParams.update({
    # Arial everywhere
    'font.family': 'Arial',
    'font.sans-serif': ['Arial'],

    'font.size': 18,
    'axes.titlesize': 30,
    'axes.labelsize': 14,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 12,
    'legend.title_fontsize': 14,

    # Keep text editable in PDF
    'pdf.fonttype': 42,
    'ps.fonttype': 42,

    # If you ever save SVG too
    'svg.fonttype': 'none',
})

# Sanity checks
if 'X_umap' not in harm.obsm:
    raise ValueError("harmonized.obsm['X_umap'] not found.")
if color_key not in adata.obs.columns:
    raise KeyError(f"{color_key} not found in bgs4.obs")
if batch_key not in harm.obs.columns:
    raise KeyError(f"{batch_key} not found in harmonized.obs")

# Restrict harmonized to CosMx batch (UMAP coords for CosMx only)
harm_cosmx_mask = harm.obs[batch_key].astype(str).values == str(target_batch)
harm_umap_cosmx = harm.obsm['X_umap'][harm_cosmx_mask, :]
harm_cosmx_names = harm.obs_names[harm_cosmx_mask]

# Align harmonized CosMx UMAP rows to bgs4 cells
if np.array_equal(harm_cosmx_names.values, adata.obs_names.values):
    umap_coords = harm_umap_cosmx
else:
    if id_col not in adata.obs.columns or id_col not in harm.obs.columns:
        raise ValueError(
            "CosMx cells in harmonized do not match bgs4.obs_names, and "
            f"'{id_col}' is missing from obs in one of the objects. "
            "Either ensure obs_names match OR set id_col to a shared unique ID."
        )

    harm_cosmx_obs = harm.obs.loc[harm_cosmx_mask, [id_col]].copy()
    harm_cosmx_obs['__umap_i'] = np.arange(harm_cosmx_obs.shape[0])

    bgs4_obs = adata.obs[[id_col]].copy()
    bgs4_obs['__bgs4_i'] = np.arange(bgs4_obs.shape[0])

    merged = bgs4_obs.merge(harm_cosmx_obs, on=id_col, how='inner')

    if merged.shape[0] == 0:
        raise ValueError(
            f"No cells matched between bgs4 and harmonized CosMx cells using id_col='{id_col}'."
        )

    if merged.shape[0] != adata.n_obs:
        print(
            f"Warning: matched {merged.shape[0]:,} / {adata.n_obs:,} bgs4 cells "
            f"to harmonized CosMx cells via '{id_col}'. Proceeding with matched subset only."
        )

    umap_coords = harm_umap_cosmx[merged['__umap_i'].values, :]
    adata = adata[merged['__bgs4_i'].values].copy()

# Keep ONLY specified cell types
labels_all = adata.obs[color_key].astype(str)
mask_keep = labels_all.isin(set(cell_type_order))

adata = adata[mask_keep].copy()
umap_coords = umap_coords[mask_keep.values, :]
labels = adata.obs[color_key].astype(str)

present = [ct for ct in cell_type_order if ct in set(labels.unique())]
label_colors = {ct: cell_type_colors.get(ct, 'lightgrey') for ct in present}
point_colors = [label_colors.get(l, 'lightgrey') for l in labels.values]

# RANDOMIZE POINT DRAW ORDER
rng = np.random.default_rng(0)
perm = rng.permutation(umap_coords.shape[0])
umap_coords = umap_coords[perm]
point_colors = np.array(point_colors)[perm]

# Plot
fig, ax = plt.subplots(figsize=(9, 5))

scat = ax.scatter(
    umap_coords[:, 0], umap_coords[:, 1],
    c=point_colors,
    s=umap_s,
    alpha=alpha,
    linewidths=0
)

# Rasterize points (keeps PDF small)
scat.set_rasterized(True)

ax.set_title(title_txt, fontsize=17)
#ax.set_xlabel("")
#ax.set_ylabel("")

# Hide tick numbers (keep clean axes)
ax.set_xticks([])
ax.set_yticks([])
ax.tick_params(bottom=False, left=False)

# REMOVE ALL SPINES / FRAME
for spine in ax.spines.values():
    spine.set_visible(False)

# Legend (outside, very close)
legend_handles = [
    mpatches.Patch(facecolor=label_colors[ct], edgecolor='none', label=ct)
    for ct in present
]

ax.legend(
    handles=legend_handles,
    loc='center left',
    bbox_to_anchor=(1.005, 0.5),
    frameon=False,
    borderaxespad=0.0,
    handlelength=1.2,
    handleheight=0.9,
)

plt.tight_layout(rect=[0, 0, 0.86, 1])

# SAVE AS PDF (Illustrator-ready: editable text + small file)
out_pdf = "GW24_CosMx_joint_embedding_celltypes_filtered.pdf"
plt.savefig(
    out_pdf,
    format="pdf",
    bbox_inches="tight",
    dpi=300,         # raster layer resolution; drop to 200 for smaller
    transparent=True
)

plt.show()
print(f"Saved: {out_pdf}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches

# CosMx-only JOINT UMAP (coords from harmonized)
# - Colored by bgs4 transferred cell types
# - ONLY the specified cell types, in the specified order
# - Custom explicit colors
# - Arial + editable text in Illustrator (PDF)
# - Rasterized points (small PDF)
# - Legend OUTSIDE, very close, with RECTANGULAR swatches
# - Hide axis tick numbers
# - Bigger title
# - REMOVE ALL SPINES / FRAME

# Objects
adata = bgs4
harm = harmonized

# Required keys / settings
color_key = 'adjusted_L3_transferred'  # labels live in bgs4
batch_key = 'batch'                    # harmonized batch column
target_batch = '0'                     # CosMx inside harmonized
id_col = 'cell_id'                     # fallback matching key if obs_names differ

# Only keep these cell types + colors
cell_type_order = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    'eMGE',
    'GPC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

# Explicit colors
cell_type_colors = {
    'eMGE': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
}

# Aesthetics
umap_s = 0.3
alpha = 1
title_txt = "2T snm3C and GW24 CosMx joint embedding"

# GLOBAL FONT + PDF SETTINGS (Arial + editable text in Illustrator)
mpl.rcParams.update({
    # Arial everywhere
    'font.family': 'Arial',
    'font.sans-serif': ['Arial'],

    'font.size': 18,
    'axes.titlesize': 30,
    'axes.labelsize': 14,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 12,
    'legend.title_fontsize': 14,

    # Keep text editable in PDF
    'pdf.fonttype': 42,
    'ps.fonttype': 42,

    # If you ever save SVG too
    'svg.fonttype': 'none',
})

# Sanity checks
if 'X_umap' not in harm.obsm:
    raise ValueError("harmonized.obsm['X_umap'] not found.")
if color_key not in adata.obs.columns:
    raise KeyError(f"{color_key} not found in bgs4.obs")
if batch_key not in harm.obs.columns:
    raise KeyError(f"{batch_key} not found in harmonized.obs")

# Restrict harmonized to CosMx batch (UMAP coords for CosMx only)
harm_cosmx_mask = harm.obs[batch_key].astype(str).values == str(target_batch)
harm_umap_cosmx = harm.obsm['X_umap'][harm_cosmx_mask, :]
harm_cosmx_names = harm.obs_names[harm_cosmx_mask]

# Align harmonized CosMx UMAP rows to bgs4 cells
if np.array_equal(harm_cosmx_names.values, adata.obs_names.values):
    umap_coords = harm_umap_cosmx
else:
    if id_col not in adata.obs.columns or id_col not in harm.obs.columns:
        raise ValueError(
            "CosMx cells in harmonized do not match bgs4.obs_names, and "
            f"'{id_col}' is missing from obs in one of the objects. "
            "Either ensure obs_names match OR set id_col to a shared unique ID."
        )

    harm_cosmx_obs = harm.obs.loc[harm_cosmx_mask, [id_col]].copy()
    harm_cosmx_obs['__umap_i'] = np.arange(harm_cosmx_obs.shape[0])

    bgs4_obs = adata.obs[[id_col]].copy()
    bgs4_obs['__bgs4_i'] = np.arange(bgs4_obs.shape[0])

    merged = bgs4_obs.merge(harm_cosmx_obs, on=id_col, how='inner')

    if merged.shape[0] == 0:
        raise ValueError(
            f"No cells matched between bgs4 and harmonized CosMx cells using id_col='{id_col}'."
        )

    if merged.shape[0] != adata.n_obs:
        print(
            f"Warning: matched {merged.shape[0]:,} / {adata.n_obs:,} bgs4 cells "
            f"to harmonized CosMx cells via '{id_col}'. Proceeding with matched subset only."
        )

    umap_coords = harm_umap_cosmx[merged['__umap_i'].values, :]
    adata = adata[merged['__bgs4_i'].values].copy()

# Keep ONLY specified cell types
labels_all = adata.obs[color_key].astype(str)
mask_keep = labels_all.isin(set(cell_type_order))

adata = adata[mask_keep].copy()
umap_coords = umap_coords[mask_keep.values, :]
labels = adata.obs[color_key].astype(str)

present = [ct for ct in cell_type_order if ct in set(labels.unique())]
label_colors = {ct: cell_type_colors.get(ct, 'lightgrey') for ct in present}
point_colors = [label_colors.get(l, 'lightgrey') for l in labels.values]

# RANDOMIZE POINT DRAW ORDER
rng = np.random.default_rng(0)
perm = rng.permutation(umap_coords.shape[0])
umap_coords = umap_coords[perm]
point_colors = np.array(point_colors)[perm]

# Plot
fig, ax = plt.subplots(figsize=(9, 5))

scat = ax.scatter(
    umap_coords[:, 0], umap_coords[:, 1],
    c=point_colors,
    s=umap_s,
    alpha=alpha,
    linewidths=0
)

# Rasterize points (keeps PDF small)
scat.set_rasterized(True)

ax.set_title(title_txt, fontsize=17)
#ax.set_xlabel("")
#ax.set_ylabel("")

# Hide tick numbers (keep clean axes)
ax.set_xticks([])
ax.set_yticks([])
ax.tick_params(bottom=False, left=False)

# REMOVE ALL SPINES / FRAME
for spine in ax.spines.values():
    spine.set_visible(False)

# Legend (outside, very close)
legend_handles = [
    mpatches.Patch(facecolor=label_colors[ct], edgecolor='none', label=ct)
    for ct in present
]

ax.legend(
    handles=legend_handles,
    loc='center left',
    bbox_to_anchor=(1.005, 0.5),
    frameon=False,
    borderaxespad=0.0,
    handlelength=1.2,
    handleheight=0.9,
)

plt.tight_layout(rect=[0, 0, 0.86, 1])

# SAVE AS PDF (Illustrator-ready: editable text + small file)
out_pdf = "20260118_GW24_CosMx_joint_embedding_celltypes_eMGE_added.pdf"
plt.savefig(
    out_pdf,
    format="pdf",
    bbox_inches="tight",
    dpi=300,         # raster layer resolution; drop to 200 for smaller
    transparent=True
)

plt.show()
print(f"Saved: {out_pdf}")